# 01 — Extraction Agents

Run Ortho, Neuro, Billing, and Baseline agents (one Gemini call per page per agent).

Start with `PAGES_TO_TEST = [1, 2]`, then set to `None` for all 10 pages.

In [ ]:
import json
from pathlib import Path

import pandas as pd

import sys
from pathlib import Path

_root = Path.cwd().resolve()
if _root.name == "notebooks":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.nb_utils import setup_project_root
from src.gemini_json import generate_json
from src.load_pages import DEFAULT_OUTPUT_DIR, iter_pages, load_corpus
from src.prompts import (
    BASELINE_SYSTEM,
    BILLING_SYSTEM,
    NEURO_SYSTEM,
    ORTHO_SYSTEM,
    page_user_message,
)
from src.schemas import (
    BaselineExtractionResult,
    BillingExtractionResult,
    NeuroExtractionResult,
    OrthoExtractionResult,
    PageRecord,
)

ROOT = setup_project_root()
DEFAULT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PAGES_TO_TEST = [1, 2]  # set to None for all pages
corpus = load_corpus()

In [ ]:
AGENTS = [
    ("ortho", ORTHO_SYSTEM, OrthoExtractionResult, "metrics"),
    ("neuro", NEURO_SYSTEM, NeuroExtractionResult, "findings"),
    ("billing", BILLING_SYSTEM, BillingExtractionResult, "lines"),
    ("baseline", BASELINE_SYSTEM, BaselineExtractionResult, "findings"),
]


def run_agent_on_page(
    name: str,
    system: str,
    model_cls,
    page: PageRecord,
    date_of_loss: str,
):
    extra = ""
    if name == "baseline":
        extra = f"date_of_loss (injury date): {date_of_loss}"
    user = page_user_message(
        page.page_number,
        page.text,
        page.date_of_service,
        extra=extra,
    )
    result, usage = generate_json(system, user, model_cls)
    return result, usage


def run_agent(name, system, model_cls, list_field: str):
    merged = []
    usage_log = []
    for page in iter_pages(corpus, PAGES_TO_TEST):
        result, usage = run_agent_on_page(
            name, system, model_cls, page, corpus.date_of_loss
        )
        items = getattr(result, list_field)
        merged.extend(items)
        usage_log.append({"page": page.page_number, **usage})
        print(f"[{name}] page {page.page_number}: {len(items)} items, tokens={usage}")
    out_path = DEFAULT_OUTPUT_DIR / f"{name}.json"
    payload = {list_field: [i.model_dump() for i in merged]}
    out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Saved {out_path}")
    return merged, usage_log

In [ ]:
ortho_metrics, _ = run_agent("ortho", ORTHO_SYSTEM, OrthoExtractionResult, "metrics")
if ortho_metrics:
    display(pd.DataFrame([m.model_dump() for m in ortho_metrics]))
else:
    print("No orthopedic metrics extracted.")

In [ ]:
neuro_findings, _ = run_agent("neuro", NEURO_SYSTEM, NeuroExtractionResult, "findings")
if neuro_findings:
    display(pd.DataFrame([f.model_dump() for f in neuro_findings]))
else:
    print("No neuro/visual findings extracted.")

In [ ]:
billing_lines, _ = run_agent("billing", BILLING_SYSTEM, BillingExtractionResult, "lines")
if billing_lines:
    display(pd.DataFrame([l.model_dump() for l in billing_lines]))
else:
    print("No billing lines extracted.")

In [ ]:
baseline_findings, _ = run_agent(
    "baseline", BASELINE_SYSTEM, BaselineExtractionResult, "findings"
)
if baseline_findings:
    display(pd.DataFrame([f.model_dump() for f in baseline_findings]))
else:
    print("No baseline findings extracted.")